# Preparation

Environment: dski

Vorbereitung des open_ragbench-Datensatzes für Experimente

PDF-Dokumente herunterladen, elementweise einlesen, vorverarbeiten und als Dataframe speichern. Abgleich und Markierung der Segmentgrenzen für Elemente in Dataframe auf Basis der Abschnitte aus den strukturierten Informationen des Datensatzes. Seiten der PDF-Dokumente in Bild-Dateien für Erkennung mittels Computer-Vision umwandeln.

Speicherung der Dataframes der vorverarbeiteten PDF-Dokumente im Ordner pdfs_df und mit hinzugefügten Embedding-Vektoren im Ordner pdfs_df_process.

## Setup

### Pakete laden

In [1]:
import config

import os
import json

import pandas as pd

from util.datasetprocessor import DatasetProcessor
from util.pdfprocessor import PDFProcessor

C:\Users\user\Documents\HOST\dskim\Master-Thesis\work\util\pdfprocessor.py:620: SyntaxWarning: invalid escape sequence '\s'
  df_doc_group_page_index_block_index['text'] = df_doc_group_page_index_block_index['text'].str.replace(r'\s{2,}', ' ', regex=True)


In [2]:
#import warnings

#warnings.filterwarnings('ignore')#action='once'

### Variablen+Funktionen

In [2]:
# DatasetProcessor
datasetProcessor = DatasetProcessor()

datasetProcessor.set_openai_client(
    api_key=config.openai_client['api_key'], 
    base_url=config.openai_client['embedding']['base_url'], 
    model_embedding='nvidia/llama-nemotron-embed-1b-v2'
)

### PDF-URLs laden

Datei mit URLs/Namen der PDF-Dokumente aus Datensatz einlesen.

In [3]:
pdf_urls = pd.read_json(f'{config.dataset["path"]}pdf_urls.json', orient='index', dtype_backend='numpy_nullable').rename(columns={0: "pdf_urls"})

## PDF-Dokumente verarbeiten

Über alle PDF-Dokumente iterieren, Dokument herunterladen und vorverarbeiten.

In [ ]:
# Dataframe für bereinigte pdf_urls erstellen
pdf_urls_clean = pd.DataFrame({'pdf_urls_clean': True}, index=pdf_urls.index)

# Ergebnisse neu generieren
reset = False

#for doc_id in pdf_urls.index:
for idx_doc in range(pdf_urls.shape[0]):#
    doc_id = pdf_urls.iloc[idx_doc].name
    print(doc_id)

    # Datei (runter)laden
    datasetProcessor.get_file(doc_id, f'{config.dataset["path"]}pdfs/')

    # rohes PDF-Dokument in Dataframe einlesen
    df_doc = PDFProcessor.doc_to_dataframe(f'{config.dataset["path"]}pdfs/{doc_id}.pdf')
    # Preprocessing Dataframe
    clean, df_doc_processed = PDFProcessor.process_dataframe(df_doc, doc_id, config.dataset['path'], reset=reset)

    # Wenn Wert der zurückgegebenen Variablen clean False
    # Dokument als nicht sauber markieren
    #if isinstance(df_doc_processed, type(None)):
    if clean == False:
        pdf_urls_clean.loc[doc_id, 'pdf_urls_clean'] = False
        print('false')
    else:
        df_doc_processed = datasetProcessor.process_document(doc_id, config.dataset['path'])
    
    # Seite des PDF-Dokuments als Bild-Dateien speichern
    PDFProcessor.pdf_pages_to_image(doc_id, config.dataset['path'])

# pdf_urls_clean als JSON-Datei speichern
#pdf_urls_clean['pdf_urls_clean'].to_json(f'{config.dataset["path"]}pdf_urls_clean.json', orient='index')